In [2]:
import sys
sys.path.append('..')

In [3]:
from src.data.load import load_ratings
import pandas as pd

In [27]:
ratings = load_ratings()

ratings["movie_id"] = ratings["movie_id"] - 1
ratings["user_id"] = ratings["user_id"] - 1
ratings =  ratings.sort_values(by='timestamp')

us_films = ratings.groupby('user_id',as_index=False)

In [28]:
train_parts = []
test_parts = []

for us_id, gr in us_films:
    if len(gr) < 3:
        continue
    split_b = int(len(gr) * 0.8)
    train_parts.append(gr.iloc[:split_b])
    test_parts.append(gr.iloc[split_b:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

In [37]:

train_df.head()

,user_id,movie_id,rating
0,0,171,5
1,0,167,5
2,0,164,5
3,0,155,4
4,0,165,5


In [40]:
from src.models.svd import SVDRecommender
from math import log2


In [34]:
model_svd=SVDRecommender(n_factors=100,n_epochs=30,random_state=42)
model_svd.fit(train_df)

In [38]:
test_likes = test_df[test_df["rating"] >= 4]
rel_movies = test_likes.groupby("user_id")["movie_id"].apply(set).to_dict()

In [39]:
seen_movies = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()

In [44]:
rec10_svd, ndcg10_svd=[],[]

In [45]:
for us_id,val in rel_movies.items():
    candidates=[m for m in train_df["movie_id"].unique().tolist() if not(m in seen_movies[us_id])]
    top_k=model_svd.recommend(us_id, top_n=10, candidate_movies=candidates)
    dcg=sum(1.0/log2(pos+1+1) for pos, ind in enumerate(top_k) if ind in val)
    idcg=sum(1.0/log2(pos+1+1) for pos in range(min(len(val),10)))
    rec10_svd.append(len(set(top_k)&val)/len(val))
    ndcg10_svd.append(dcg / idcg if idcg > 0 else 0.0)

In [46]:
print(f'SVD Recall@10: {sum(rec10_svd)/len(rec10_svd)}')
print(f'SVD NDCG@10: {sum(ndcg10_svd)/len(ndcg10_svd)}')

SVD Recall@10: 0.04560274814172815
SVD NDCG@10: 0.06615482812532669


In [51]:
from TTR_model import TwoTowerRetrieval
import torch
import torch.nn as N
import torch.nn.functional as F
from src.utils.config import EMBS_ITEM, EMBS_USER, EMBS
from data_split_and_history import MovieRetrievalDataset, prep_hist_n_data
from torch.utils.data import DataLoader
from src.evaluation.eval_retrieval import evaluation_retrieval
from src.evaluation.eval_retrieval import evaluation_retrieval

In [ ]:
prep_feature_embs = torch.load(EMBS_ITEM, weights_only=True)
prep_user_embs = torch.load(EMBS_USER,weights_only=True)
train_data, test_data =prep_hist_n_data(prep_feature_embs)

In [49]:
device='cpu'
num_epochs = 100
lr=1e-3
temperature=0.07
num_us=943
num_items=1682
hist_dim=384
lay_sz=64
id_lay_sz=16
output_sz=64
proj_hist_sz=64
features=prep_feature_embs.shape[1]

In [55]:
model_TTR=TwoTowerRetrieval(temperature,prep_user_embs,prep_feature_embs,num_us,num_items,id_lay_sz,lay_sz,output_sz,hist_dim,proj_hist_sz,features).to(device)
model_TTR.load_state_dict(torch.load(EMBS/'TTR_weights.pt',weights_only=True))

<All keys matched successfully>

In [56]:
rec10_ttr, ndcg10_ttr=evaluation_retrieval(model_TTR,10,train_data,test_data,num_items,device)
print(f"Recall@10 TTR: {rec10_ttr}")
print(f'NDGC@10 TTR: {ndcg10_ttr}')

Recall@10 TTR: 0.05764993358573007
NDGC@10 TTR: 0.06012345206996189
